# 1 · Quick smoke test — one model

Run this **before** the full demo to catch any load / chat-template / thinking-mode
problem in ~2 minutes instead of mid-run. Loads a single model (4-bit) and answers
a few inline questions.

Pick which model to test in the last cell (`TEST_SPEC`): Qwen3.5-4B, Phi-3.5-mini, etc.

Pass criteria: no exception, non-empty outputs, a letter / number is extracted.
**Settings → GPU (T4) + Internet On**; add the `HF_TOKEN` secret.

In [ ]:
REPO_URL = "https://github.com/ThuongHong/science-of-benchmarking-demo.git"
import os, sys, subprocess
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
WORK = f"/kaggle/working/{REPO_DIR}" if os.path.isdir("/kaggle") else REPO_DIR
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, "src")
!pip install -q -U transformers accelerate bitsandbytes sentencepiece
print("cwd:", os.getcwd())

In [ ]:
# HF auth (Qwen is usually ungated, but a token never hurts)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("HF login OK")
except Exception as e:
    print("skipping HF login:", e)

In [ ]:
# A couple of inline items (no dataset download needed)
items = [
    {"id": "t_mmlu_0", "benchmark": "mmlu", "subject": "test",
     "question": "What is the capital of France?",
     "choices": ["Berlin", "Paris", "Rome", "Madrid"],
     "answer_index": 1, "answer": "B"},
    {"id": "t_mmlu_1", "benchmark": "mmlu", "subject": "test",
     "question": "Which gas do plants absorb during photosynthesis?",
     "choices": ["Oxygen", "Nitrogen", "Carbon dioxide", "Hydrogen"],
     "answer_index": 2, "answer": "C"},
    {"id": "t_gsm_0", "benchmark": "gsm8k",
     "question": "Tom has 3 boxes with 4 apples each. He eats 2. How many are left?",
     "answer": "10", "solution": "3*4=12; 12-2=10"},
    {"id": "t_gsm_1", "benchmark": "gsm8k",
     "question": "A book costs $7. How much do 5 books cost?",
     "answer": "35", "solution": "7*5=35"},
]
len(items)

In [ ]:
import config, models
from metrics import extract_letter, extract_number_robust

# Pick the model to test. Examples:
TEST_SPEC = config.QWEN_STRONG     # Qwen3.5-4B
# TEST_SPEC = config.PEERS[0]      # Phi-3.5-mini
# TEST_SPEC = config.PEERS[1]      # Qwen2.5-3B
# TEST_SPEC = config.GEMMA_SYSTEMS[1]   # Gemma-4 E4B

runner = models.make_system(TEST_SPEC)
print("loaded:", runner.name, runner.model_id)

ok = True
for it in items:
    out = runner.predict(it)
    got = (extract_letter(out) if it["benchmark"] == "mmlu"
           else extract_number_robust(out))
    ok = ok and bool(out.strip()) and got is not None
    print(f"\n[{it['id']}] gold={it['answer']} extracted={got}")
    print("  raw:", repr(out[:300]))

runner.unload()
print("\nSMOKE TEST:", "PASS ✅ (non-empty + extractable)" if ok else "FAIL ❌")